In [1]:
import numpy as np
import pandas as pd
import os
import csv

In [ ]:
"""
- All Triangle connections are in a "flipped" order. Two triangles with the same orientation can't connect on any edges. 
- This is orientation will be included in the triangle creation.
- This also means we can create a "true" orientation for triangles when placed/joined: 
    Default Orientation: Point Facing Upwards
- On this Default Orientation define the edges in the order: Left(0), Right(1), Bottom(2)
- And if you look at some examples of this orientation you will see that it is the same orientations that join together.
    For Example: The Bottoms(2) touch, the Lefts(0) touch, and the Rights(1) touch.
"""

class Triangle:

    # This flag is used for checking/searching through Pieces.
    checked = False

    def __init__(self, id, orientation, edge_0=None, edge_1=None, edge_2=None, unit=None):
        self.id = id
        self.orientation = orientation
        self.edges = [edge_0, edge_1, edge_2]
        self.unit = unit
        self.connectEdges()

    def printEdge(self, n):
        if self.edges[n] is None:
            return "None"
        else:
            return f"{self.edges[n].id}"

    def __str__(self):
        return f"Triangle {self.id} | [{self.printEdge(0)}, {self.printEdge(1)}, {self.printEdge(2)} | {self.orientation}]"
    
    def __repr__(self):
        return f"Triangle {self.id} | [{self.printEdge(0)}, {self.printEdge(1)}, {self.printEdge(2)}] | Unit: {self.unit}"
    
    def setEdge(self, n, edge):
        self.edges[n] = edge

    def connectEdges(self):
        for n, edge in enumerate(self.edges):
            if edge:
                if self.orientation == edge.orientation: raise ValueError(f"There is an error with connecting {self.id} {self.unit} with {edge.id} {edge.unit}")
                edge.setEdge(n, self)
            

class Piece:

    rotation = 1
    flipped = False
    solvedTemp = False
    placed = False

    def __init__(self, fileLocation=None, name=None):
        self.fileLocation = fileLocation
        if fileLocation is None:
            self.triangleSet = {1: Triangle(1, True, unit=name)}
        else:
            self.name = os.path.splitext(os.path.basename(fileLocation))[0]
            self.createPiece()

    def __repr__(self):
        return f"Piece {self.name} | {self.placed}"
        
    def createPiece(self):
        self.triangleSet = {None:None}
        triangleInfo = pd.read_csv(self.fileLocation, dtype={"id":"Int64", "orientation":bool, "edge_0":"Int64", "edge_1":"Int64", "edge_2":"Int64"})
        triangleInfo = triangleInfo.replace({np.nan: None}) #triangleInfo.where(pd.notnull(triangleInfo), None)
        for n in range(len(triangleInfo)):
            triangle = triangleInfo.iloc[n].copy()
            if triangle.edge_0 and triangle.edge_0 >= triangle.id: triangle.edge_0 = None
            if triangle.edge_1 and triangle.edge_1 >= triangle.id: triangle.edge_1 = None
            if triangle.edge_2 and triangle.edge_2 >= triangle.id: triangle.edge_2 = None
            #print(triangle.edge_0, triangle.edge_1, triangle.edge_2)
            self.triangleSet[triangle.id] = Triangle(id = triangle.id,
                                               orientation = triangle.orientation,
                                               edge_0 = self.triangleSet[triangle.edge_0],
                                               edge_1 = self.triangleSet[triangle.edge_1],
                                               edge_2 = self.triangleSet[triangle.edge_2],
                                               unit = self.name
                                              )
        self.triangleSet.pop(None)

    def rotatePiece(self):
        # Let's always rotate once clockwise (There are 6 total rotations before back to start).
        self.rotation += 1
        for triangle in self.triangleSet.values():
            triangle.orientation = not triangle.orientation
            # Each orientation changes the edge assignments by subtracting one.
            triangle.edges[0], triangle.edges[1], triangle.edges[2] = triangle.edges[1], triangle.edges[2], triangle.edges[0] 

    def flipPiece(self):
        self.flipped = not self.flipped
        for triangle in self.triangleSet.values():
            triangle.edges[0], triangle.edges[1], triangle.edges[2] = triangle.edges[1], triangle.edges[0], triangle.edges[2] 

    def triangleCount(self):
        return  len(self.triangleSet)

In [ ]:

def findNextPiece(startingPoint, piece):
    triangles = []
    for triangle in piece:
        triangles.append((triangle[0]+startingPoint[0], triangle[1]+startingPoint[1],((triangle[2]+startingPoint[2])%2)+1))
    return triangles

def findValidPlacements(board, piece):
    # We want to loop through all spots in the board and return a list of all valid starting locations for the given piece.
    validSpots = []
    for spot in board.values():
        trianglesToCheck = findNextPiece(spot, piece)
        # Below calculates the intersection of the two sets.
        if len(set(trianglesToCheck) & set(board.keys())) == piece.triangleCount():
            validSpots.append(spot)
    return validSpots

def placePiece(board, locations, piece):
    for spot in locations:
        board[spot][3] = piece.name
    return board


In [ ]:
def setupGame():
    # Create Board
    with open('Resources2/Board.csv', mode='r', newline='') as file:
        reader = csv.reader(file)
        # Convert the CSV data into a list of tuples
        boardInfo = [(int(row[0]), int(row[1]), int(row[2]), None) for row in reader]
    board = {str(triangle):triangle for triangle in boardInfo}
    boardIndx = {indx+1:str(triangle) for indx, triangle in enumerate(reader)}
    # Loop Through and Create Pieces
    folderPath = "Resources/GamePieces/"
    pieces = {}
    for fileName in os.listdir(folderPath):
        filePath = os.path.join(folderPath, fileName)
        if os.path.isfile(filePath):
            pieces[fileName] = (Piece(filePath))
    # Create the Seven Triangles (Obstacles) that are set by the Dice.
    obstacles = []
    for setPiece in range(1,8):
        obstacles.append(Piece(name="Obstacle" + str(setPiece)))
    return board, boardIndx, pieces, obstacles

def reorderPices(pieces):
    return [pieces["DarkGreen5.csv"], pieces["Lavender5.csv"], pieces["LightGreen5.csv"], pieces["Red5.csv"], 
            pieces["Purple4.csv"], pieces["Orange4.csv"], pieces["Pink4.csv"], pieces["Blue3_1.csv"], pieces["Blue3_2.csv"], pieces["Yellow2.csv"], pieces["DarkBlue1.csv"]]

def rollDice(board, boardIndx, obstacles):
    rand1 = np.random.choice([12, 13, 23, 24, 32, 33, 41, 42])
    randIndx1 = boardIndx[rand1]
    placePiece(board, [randIndx1], obstacles[0])
    rand2 = np.random.choice([2, 4, 7, 8, 9, 11, 16, 17])
    randIndx2 = boardIndx[rand2]
    placePiece(board, [randIndx2], obstacles[1])
    rand3 = np.random.choice([25, 26, 36, 37, 38, 40, 45, 47])
    randIndx3 = boardIndx[rand3]
    placePiece(board, [randIndx3], obstacles[2])
    rand4 = np.random.choice([10,27,31])
    randIndx4 = boardIndx[rand4]
    placePiece(board, [randIndx4], obstacles[3])
    rand5 = np.random.choice([18, 22, 39])
    randIndx5 = boardIndx[rand5]
    placePiece(board, [randIndx5], obstacles[4])
    rand6 = np.random.choice([1, 5, 15, 34, 44, 48])
    randIndx6 = boardIndx[rand6]
    placePiece(board, [randIndx6], obstacles[5])
    rand7 = np.random.choice([19, 20, 21, 28, 29, 30])
    randIndx7 = boardIndx[rand7]
    placePiece(board, [randIndx7], obstacles[6])
    print(f"The Dice Rolls are {rand1}, {rand2}, {rand3}, {rand4}, {rand5}, {rand6}, {rand7}")

In [23]:
Board = [
    (1,6,1),
    (2,5,1),
    (2,6,2),
    (2,7,1),
]

Dictionary = {str(triangle):triangle for triangle in Board}

PieceDNF = [
    (0,0,2),
    (0,1,1),
    (1,1,2)
]

PieceFit = [
    (0,0,1),
    (1,0,2),
    (1,-1,1)
]
trianglesToCheck = findNextPiece((1,6,1), PieceFit)
print(trianglesToCheck)

results = [0 if str(triangle) in Dictionary else 1 for triangle in trianglesToCheck]
fits = np.sum(results) == 0
fits

[(1, 6, 1), (2, 6, 2), (2, 5, 1)]


np.True_

In [6]:
board, boardIndx, pieces, obstacles = setupGame() # Build the Board and Pieces.
pieces = reorderPices(pieces)
# rollDice(board, obstacles) # Roll the Dice and Add the Obstacles to the Board.




In [ ]:
def solvePuzzle(board, pieces):
    piece = pieces.pop(0)
    validSpots = findValidPlacements(board, piece)
    for spot in validSpots:
        pieceLocations = [() for ]
        board = placePiece(board, pieceLocations, piece)
        # Call itself again
        # Is the return call a success?
    # TODO Rotate / Flip the Piece
    

In [ ]:
def solvePuzzle(board, pieces):
    """ 
    Recursive Function that uses the board and available pieces as it's input
    In each call it takes a piece and brute force tries it at every triangle. Then rotates it. 
    If it ever succeeds it calls itself with 1 less piece. If it fails it returns itself witht the piece
    still available, then it continues the previous pieces brute force.
    """
    for piece in pieces:
        if not piece.placed:
            break
    if piece.placed:
        if board.isSolved():
            board.solvedTemp = True
            return board, pieces
        return board, pieces
    for indx in range(1, board.triangleCount() + 1):
        if checkValidPlacement(board, indx, piece):
            if piece.name == "LightGreen5": print("Placed ", indx)
            # print(piece, indx)
            piece.resetTriangleSearch() # Current limitation is that this doesn't happen for valid returns
            placePiece(board, indx, piece)
            piece.placed = True
            piece.resetTriangleSearch()
            board, pieces = solvePuzzle(board, pieces)
            if board.solvedTemp:
                return board, pieces
            removePiece(board, piece)
            piece.placed = False
    piece.rotatePiece()
    if piece.name == "LightGreen5": print("Rotated ", piece.rotation)
    if piece.rotation <= 6:
        board, pieces = solvePuzzle(board, pieces)
        if board.solvedTemp:
            return board, pieces
        # removePiece(board, piece)
        # piece.placed = False
    piece.rotation = 1
    if not piece.flipped:
        if piece.name == "LightGreen5": print("Flipped ", piece.flipped)
        piece.flipPiece()
        board, pieces = solvePuzzle(board, pieces)
        if board.solvedTemp:
            return board, pieces
        piece.flipPiece()
    return board, pieces

In [7]:
data

[(1, 6, 1),
 (2, 5, 1),
 (2, 6, 1),
 (2, 7, 1),
 (3, 1, 2),
 (3, 2, 1),
 (3, 3, 2),
 (3, 4, 1),
 (3, 5, 2),
 (3, 6, 1),
 (3, 7, 2),
 (3, 8, 1),
 (3, 9, 2),
 (3, 10, 1),
 (3, 11, 2),
 (4, 2, 2),
 (4, 3, 1),
 (4, 4, 2),
 (4, 5, 1),
 (4, 6, 2),
 (4, 7, 1),
 (4, 8, 2),
 (4, 9, 1),
 (4, 10, 2),
 (5, 2, 1),
 (5, 3, 2),
 (5, 4, 1),
 (5, 5, 2),
 (5, 6, 1),
 (5, 7, 2),
 (5, 8, 1),
 (5, 9, 2),
 (5, 10, 1),
 (6, 1, 1),
 (6, 2, 2),
 (6, 3, 1),
 (6, 4, 2),
 (6, 5, 1),
 (6, 6, 2),
 (6, 7, 1),
 (6, 8, 2),
 (6, 9, 1),
 (6, 10, 2),
 (6, 11, 1),
 (7, 5, 2),
 (7, 6, 1),
 (7, 7, 2),
 (8, 6, 2)]

In [35]:
from timeit import time

n = 1000000
total = 0
for i in range(n):
    start = time.time()
    fits = pd.Series.sum(results) == 0
    end = time.time()
    total += end-start
total

AttributeError: 'list' object has no attribute '_min_count_stat_function'

In [ ]:
    trianglesToCheck = findNextPiece((1,6,1), PieceFit)
    results = [0 if str(triangle) in Dictionary else 1 for triangle in trianglesToCheck]
    fits = np.sum(results) == 0